In [ ]:
#INSTALL GRADIO

#DOCUMENTS
#https://github.com/gradio-app/gradio
#https://huggingface.co/learn/llm-course/chapter9/1

#INSTALL
#pip install gradio

<h3>Whisper</h3>

In [29]:
import socket
import numpy as np
import torch
import gradio as gr
from transformers import WhisperProcessor, WhisperForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

#Erillaisia malleja ovat:
# model_name = "openai/whisper-tiny" - kevyt nopea ja epäluotettava suomeksi (Englanniksi hyvä)
# model_name = "openai/whisper-base" - kevyt ja nopea, mutta ei niin tarkka suomeksi.
# model_name = "openai/whisper-small" - kevyt, mutta bufferointi kestää n. 4s
# model_name = "openai/whisper-medium"  - suhteellisen raskas ajo omalla vastuulla
# model_name = "openai/whisper-large-v3" - raskas

model_name = "openai/whisper-bas"

print(f"Using device: {device}")

Using device: cpu


<b>Testailua:</b>

- tiny: todella nopea reaaliajassa, onglema voi olla suomenkieli alle 1s
- base: edelleen todella nopea prosessointi alle 1.5s
- small: liian hidas nopeaan prosessointiin, yhden komennon läpivienti kestää ~4s

In [30]:
print(f"Aloitetaan mallin lataus {model_name} laitteelle {device}...")
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)
model.eval()

Aloitetaan mallin lataus openai/whisper-tiny laitteelle cpu...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 384, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(384, 384, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 384)
      (layers): ModuleList(
        (0-3): 4 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=384, out_features=384, bias=False)
            (v_proj): Linear(in_features=384, out_features=384, bias=True)
            (q_proj): Linear(in_features=384, out_features=384, bias=True)
            (out_proj): Linear(in_features=384, out_features=384, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
   

In [53]:
audio_buffer = np.array([], dtype=np.float32)
TARGET_SR = 16000
MIN_RMS = 0.012
MIN_BUFFER_SECONDS = 0.5
MAX_BUFFER_SECONDS = 1.3

#Muuntaa äänidatan yhdeksi monokanavaiseksi (1D) float32-taulukoksi välille [-1.0, 1.0].
def to_mono_float32(audio_data: np.ndarray) -> np.ndarray:
    # Varmistetaan, että syöte on NumPy-taulukko (vaikka annettaisiin esim. lista)
    arr = np.asarray(audio_data)

    # Jos taulukko on täysin tyhjä (0 näytettä), palautetaan tyhjä float32-taulukko
    if arr.size == 0:
        return np.array([], dtype=np.float32)

    # Jos ääni on monikanavainen (esim. stereo 2D), otetaan viimeisen akselin keskiarvo,
    # jolloin tuloksena on yksi mono-kanava (1D)
    if arr.ndim > 1:
        arr = arr.mean(axis=-1)

    # Jos näytteet ovat kokonaislukuja (esim. int16), skaalataan ne float32-muotoon
    # välille noin [-1.0, 1.0] jakamalla kyseisen kokonaislukutyypin maksimiarvolla
    if np.issubdtype(arr.dtype, np.integer):
        max_val = float(np.iinfo(arr.dtype).max)
        arr = arr.astype(np.float32) / max_val
    # Muussa tapauksessa (esim. jo valmiiksi liukulukuja) muunnetaan suoraan float32-tyypiksi
    else:
        arr = arr.astype(np.float32)

    # Rajataan arvot tiukasti välille [-1.0, 1.0], jotta mahdolliset yliohjautuvat piikit
    # tai pienet pyöristyserot eivät mene sallitun äänialueen ulkopuolelle
    return np.clip(arr, -1.0, 1.0)

In [54]:
def transcribe_stream(stream_tuple, saved_text: str):
    global audio_buffer
    current_text = saved_text or ""

    if stream_tuple is None:
        return current_text, current_text

    sr, raw_audio = stream_tuple
    mono = to_mono_float32(raw_audio)
    if mono.size == 0:
        return current_text, current_text

    if sr != TARGET_SR:
        duration = mono.shape[0] / float(sr)
        target_len = max(1, int(round(duration * TARGET_SR)))
        old_x = np.linspace(0.0, 1.0, num=mono.shape[0], endpoint=False)
        new_x = np.linspace(0.0, 1.0, num=target_len, endpoint=False)
        mono = np.interp(new_x, old_x, mono).astype(np.float32)

    audio_buffer = np.concatenate([audio_buffer, mono])

    max_samples = int(MAX_BUFFER_SECONDS * TARGET_SR)
    if audio_buffer.shape[0] > max_samples:
        audio_buffer = audio_buffer[-max_samples:]

    min_samples = int(MIN_BUFFER_SECONDS * TARGET_SR)
    if audio_buffer.shape[0] < min_samples:
        return current_text, current_text

    rms = float(np.sqrt(np.mean(np.square(audio_buffer))))
    if rms < MIN_RMS:
        return current_text, current_text

    inputs = processor(
        audio_buffer,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
        truncation=True,
        padding="longest",
        return_attention_mask=True,
    ).to(device)

    if "input_features" in inputs:
        inputs["input_features"] = inputs["input_features"].to(dtype=model.dtype)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            language="fi", #language="en" kokeilu englanniksi
            task="transcribe",
            return_timestamps=False,
        )

    transcription = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    audio_buffer = np.array([], dtype=np.float32)

    if not transcription:
        return current_text, current_text

    print(f"[WHISPER] RMS={rms:.4f} -> '{transcription}'")
    updated_text = f"{current_text} {transcription}".strip() if current_text else transcription
    return updated_text, updated_text

In [55]:
#Tyhjentää bufferin/tekstikentän.
def clear_all():
    global audio_buffer
    audio_buffer = np.array([], dtype=np.float32)
    return "", ""

<h3>Gradio</h3>

In [56]:
with gr.Blocks(title="Whisper Reaaliaikatesti") as demo:
    gr.Markdown("## Whisper testi gradiolla")
    gr.Markdown("Kuinka käyttää: ")
    gr.Markdown("1. Käynnistä nauhoitus ja salli selaimessa.")
    gr.Markdown("2. Puhu mikrofooniin yksittäisiä sanoja. Ohjelman olisi tarkoitus miltei reaaliajassa kirjoitta ylös puhumasi sanat.")
    transcript_state = gr.State("")

    with gr.Row():
        mic_stream = gr.Audio(
            sources=["microphone"],
            streaming=True,
            recording=True,
            type="numpy",
            label="Ääni",
        )
        transcript_box = gr.Textbox(
            label="Tunnistettu teksti",
            lines=4,
            interactive=False,
        )

    clear_btn = gr.Button("Tyhjennä teksti")

    mic_stream.stream(
        fn=transcribe_stream,
        inputs=[mic_stream, transcript_state],
        outputs=[transcript_box, transcript_state],
    )

    clear_btn.click(
        fn=clear_all,
        inputs=None,
        outputs=[transcript_box, transcript_state],
    )
    
gr.close_all()
demo.launch()

Closing server running on port: 7866
* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


<h3>Yhteenvetoja:</h3>

- Englanniksi toimii tiny jo todella tehokkaasti tunnistamaan yksittäisiä komentoja.
- Suomeksi base on ok.

- Jos nopeatempoisessa ohjelmistossa otettaisiin koko ajan vastaan komentoja ja niitä toistettaisiin useita kertoja, se voisi toimia.